# 第 17 课｜从零组装微型 GPT 并进行字符级预训练

**从随机参数开始，让一个完整的 decoder-only Transformer 学习预测下一个字符。**

这一课把第 13–16 课的词表、Embedding、交叉熵、数据批次、位置表示、多头因果注意力、LayerNorm 和残差连接组合起来。
最终得到的不是现成模型接口，而是你能够逐层解释、训练、验证、保存和生成的教学模型。

| 本课产物 | 说明 |
|---|---|
| 模型 | 两层 Pre-LN Transformer，字符词表，独立词表输出头 |
| 实验 | 模板语料训练/验证、按验证损失选模型、逐字符续写 |
| 材料 | 本 Notebook + 同名主题的独立 `.py` 脚本；互不依赖 |
| 硬件 | CPU 即可；有可用 CUDA 时可以选择显卡 |

**这是微型自回归预训练实验，不是复现商用 GPT 的规模、数据、分词或完整训练配方。**
模型没有指令微调，不是聊天助手，也没有获得通用推理、编程或 Agent 能力。

## 学习顺序与完成标准

先看数据和张量怎样流动，再阅读训练/生成代码，最后完成三个独立练习。
建议分两次学习：第一次做到“损失和因果性检查”，第二次完成训练、生成与练习。

完成时，应能解释这五个问题：

1. `[B,T]` 的编号怎样变成 `[B,T,V]` 的词表 logits？
2. 为什么训练可以同时预测所有位置，却仍不能读取未来？
3. PAD 的注意力遮挡与标签 `-100` 分别解决什么问题？
4. 生成时为什么只取 `logits[:, -1, :]`，为什么不做 `backward()`？
5. 为什么保存“最低验证损失的模型”，与保存“最后一轮模型”可能不同？

本讲义的主体定义与脚本一致；示例、练习和参考答案只在讲义中展开。
**练习检查默认关闭，运行所有单元成功不代表已经独立完成练习。**

## 0. 环境与约定

继续使用前几课已经能运行的 Python/PyTorch 环境。没有新增第三方依赖，不下载权重或语料。
缺少 PyTorch 时，可在当前内核运行 `%pip install torch`；需要 GPU 安装选项时查阅
[PyTorch 官方安装说明](https://pytorch.org/get-started/locally/)。不要为了本课破坏已经可用的 CUDA 环境。

下面使用 `@dataclass` 把模型配置放在一个对象里。`GPTConfig(...)` 生成带有同名属性的配置，
`__post_init__()` 在构造后检查取值，`asdict(cfg)` 把配置转成可保存的字典。
它只是管理超参数，不是另一种神经网络层。

默认 `DEVICE="cpu"`。把它改成 `"auto"` 会在 CUDA 可用时使用显卡。不同设备、库版本不保证逐位一致，固定随机种子也不能消除所有差异。[9]

In [1]:
from __future__ import annotations

import argparse
from dataclasses import asdict, dataclass
from functools import partial
import hashlib
import json
import math
from pathlib import Path
import random
import tempfile

import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNK_ID, BOS_ID, EOS_ID, PAD_ID = range(4)
IGNORE_INDEX = -100


def positive_int(value: int, name: str) -> None:
    if type(value) is not int or value <= 0:
        raise ValueError(f"{name} 必须是正整数，不接受布尔值")


def choose_device(requested: str = "cpu") -> torch.device:
    if requested not in {"cpu", "cuda", "auto"}:
        raise ValueError("device 必须是 cpu、cuda 或 auto")
    if requested == "auto":
        requested = "cuda" if torch.cuda.is_available() else "cpu"
    if requested == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA 不可用；请先用 --device cpu，或检查已有 CUDA 环境")
    return torch.device(requested)


def set_seed(seed: int) -> None:
    if type(seed) is not int or not 0 <= seed < 2**32:
        raise ValueError("seed 必须是 [0,2**32) 内的整数")
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # 固定种子有助于复现；不同设备、PyTorch 版本不保证逐位一致。


@dataclass
class GPTConfig:
    vocab_size: int
    max_len: int = 64
    d_model: int = 64
    n_heads: int = 4
    n_layers: int = 2
    ff_dim: int = 256
    dropout: float = 0.1

    def __post_init__(self) -> None:
        for name in ("vocab_size", "max_len", "d_model", "n_heads", "n_layers", "ff_dim"):
            positive_int(getattr(self, name), name)
        if self.vocab_size <= len(SPECIAL_TOKENS):
            raise ValueError("词表除了 4 个特殊 token，还必须有普通字符")
        if self.d_model % self.n_heads:
            raise ValueError("d_model 必须能被 n_heads 整除")
        if (isinstance(self.dropout, bool) or not isinstance(self.dropout, (int, float))
                or not math.isfinite(self.dropout) or not 0 <= self.dropout < 1):
            raise ValueError("dropout 必须在 [0,1) 内")

In [2]:
torch.set_num_threads(1)
DEVICE = choose_device("cpu")
set_seed(17)
print("PyTorch：", torch.__version__)
print("当前设备：", DEVICE)
print("CUDA 可用：", torch.cuda.is_available())
print("工作目录：", Path.cwd())

PyTorch： 2.10.0+cpu
当前设备： cpu
CUDA 可用： False
工作目录： /mnt/data/_lesson17_work/notebook_run


## 1. 训练什么数据？先切分，再建词表

内置语料由 8 个主语 × 2 个时间 × 6 个活动组成，一共 96 条不同句子，例如：

```text
我今天正在学习数学。
小明明天正在训练模型。
老师今天正在整理笔记。
```

它们是为了观察机制而构造的教学模板。训练/验证的句子不完全相同，但共享模板和词语，
所以验证集成绩不能解释为在自然语言任务上的广泛泛化能力。

流程：清除空行和完全重复的整句 → 按整句切分约 80%/20% → 只用训练侧字符建立词表。
验证侧的新字符编码为 `<unk>`，不会补进训练词表；代码会记录未知字符数。
内部空格保留，句子两端空白会去掉。每句话是独立样本，不拼接到别句话的上下文中。

In [3]:
def demo_lines() -> list[str]:
    """96 条原创模板句；目的只是验证训练流程，不是中文能力基准。"""
    subjects = ["我", "你", "小明", "小红", "同学", "老师", "我们", "他们"]
    times = ["今天", "明天"]
    activities = ["学习数学", "编写程序", "阅读故事", "整理笔记", "检查数据", "训练模型"]
    return [s + t + "正在" + a + "。" for s in subjects for t in times for a in activities]


def ensure_demo_file(path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with path.open("x", encoding="utf-8") as handle:
            handle.write("\n".join(demo_lines()) + "\n")
    except FileExistsError:
        pass
    return path


def clean_texts(lines: list[str]) -> list[str]:
    if not isinstance(lines, list) or any(not isinstance(t, str) for t in lines):
        raise ValueError("语料必须是字符串列表")
    texts, seen = [], set()
    for line in lines:
        text = line.strip()
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts


def split_texts(texts: list[str], seed: int = 17) -> tuple[list[str], list[str]]:
    if (not isinstance(texts, list) or len(texts) < 5
            or any(not isinstance(t, str) or not t for t in texts)):
        raise ValueError("清理后至少需要 5 条不同的非空文本")
    if len(set(texts)) != len(texts):
        raise ValueError("请先按完整文本去重，再切分")
    shuffled = list(texts)
    random.Random(seed).shuffle(shuffled)
    n_val = max(1, len(texts) // 5)
    return shuffled[n_val:], shuffled[:n_val]


def build_vocab(train_texts: list[str]) -> tuple[dict[str, int], list[str]]:
    if not train_texts or any(not isinstance(t, str) for t in train_texts):
        raise ValueError("训练文本必须是非空字符串列表")
    # 排序固定编号顺序；只能使用训练侧字符。
    chars = sorted(set("".join(train_texts)))
    id_to_token = SPECIAL_TOKENS + chars
    return {token: i for i, token in enumerate(id_to_token)}, id_to_token


def validate_vocab(id_to_token: list[str]) -> None:
    if (not isinstance(id_to_token, list) or len(id_to_token) <= 4
            or id_to_token[:4] != SPECIAL_TOKENS
            or any(not isinstance(t, str) for t in id_to_token)
            or any(len(t) != 1 for t in id_to_token[4:])
            or len(set(id_to_token)) != len(id_to_token)):
        raise ValueError("词表必须按固定顺序包含四个特殊 token，之后是互不重复的单个字符")


def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    return [token_to_id.get(char, UNK_ID) for char in text]

In [4]:
example_texts = clean_texts(demo_lines())
train_texts, validation_texts = split_texts(example_texts, seed=17)
token_to_id, id_to_token = build_vocab(train_texts)
print("训练 / 验证：", len(train_texts), len(validation_texts))
print("完整文本交集：", set(train_texts) & set(validation_texts))
print("词表大小：", len(id_to_token))
print("特殊 token：", id_to_token[:4])
print("训练样例：", train_texts[:3])
print("验证样例：", validation_texts[:3])

训练 / 验证： 77 19
完整文本交集： set()
词表大小： 41
特殊 token： ['<unk>', '<bos>', '<eos>', '<pad>']
训练样例： ['老师今天正在整理笔记。', '小明今天正在检查数据。', '我今天正在检查数据。']
验证样例： ['他们今天正在阅读故事。', '同学明天正在阅读故事。', '你明天正在学习数学。']


## 2. 自回归目标：移位只做一次

完整句子先变成 `[<bos>, 字1, 字2, …, <eos>]`，然后：

```python
input_ids = full_ids[:-1]
targets   = full_ids[1:]
```

例如“我今天”的输入是 `[<bos>, 我, 今, 天]`，标签是 `[我, 今, 天, <eos>]`。
本课 `Dataset` 已经完成移位；`TinyGPT.forward()` 和损失函数都不再移位。
不要把某些第三方模型“内部自动移位”的接口约定直接套到这里。

训练时每个位置使用真实前缀，这通常称为 teacher forcing。所有位置可并行计算，
但注意力 mask 保证位置 `t` 只读取 `≤t` 的输入。**并行计算不意味着允许偷看标签。**

In [5]:
class CharDataset(Dataset):
    def __init__(self, texts: list[str], token_to_id: dict[str, int], max_len: int) -> None:
        positive_int(max_len, "max_len")
        if not isinstance(texts, list) or not texts:
            raise ValueError("数据集需要非空文本列表")
        self.sequences = []
        for index, text in enumerate(texts):
            ids = [BOS_ID] + encode(text, token_to_id) + [EOS_ID]
            # 输入和目标比完整序列各少一位；不允许静默截断。
            if len(ids) - 1 > max_len:
                raise ValueError(f"第 {index + 1} 条文本需 {len(ids)-1} 个输入位置，超过 max_len={max_len}；请手动分段或调大上下文")
            self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self) -> int:
        return len(self.sequences)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        ids = self.sequences[index]
        return ids[:-1], ids[1:]


def collate_batch(samples: list[tuple[torch.Tensor, torch.Tensor]]) -> dict[str, torch.Tensor]:
    if not samples:
        raise ValueError("不能整理空批次")
    xs, ys = [], []
    for x, y in samples:
        if (x.ndim != 1 or x.numel() == 0 or x.shape != y.shape
                or x.dtype != torch.long or y.dtype != torch.long):
            raise ValueError("每条样本的输入、标签必须是等长非空 long 向量")
        xs.append(x)
        ys.append(y)
    return {
        "input_ids": pad_sequence(xs, batch_first=True, padding_value=PAD_ID),
        "targets": pad_sequence(ys, batch_first=True, padding_value=IGNORE_INDEX),
    }


def make_loader(dataset: Dataset, batch_size: int, shuffle: bool = False, seed: int = 17) -> DataLoader:
    positive_int(batch_size, "batch_size")
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
                      num_workers=0, collate_fn=collate_batch,
                      generator=torch.Generator().manual_seed(seed))

In [6]:
preview_data = CharDataset(["我今天", "我今天正在学习数学。"], token_to_id, max_len=64)
preview_batch = collate_batch([preview_data[0], preview_data[1]])
print("输入形状：", tuple(preview_batch["input_ids"].shape))
print("标签形状：", tuple(preview_batch["targets"].shape))
print("短句输入：", [id_to_token[i] for i in preview_batch["input_ids"][0].tolist()])
print("短句标签：", [id_to_token[i] if i != IGNORE_INDEX else "忽略" for i in preview_batch["targets"][0].tolist()])
print("有效目标数：", int((preview_batch["targets"] != IGNORE_INDEX).sum()))

输入形状： (2, 11)
标签形状： (2, 11)
短句输入： ['<bos>', '我', '今', '天', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
短句标签： ['我', '今', '天', '<eos>', '忽略', '忽略', '忽略', '忽略', '忽略', '忽略', '忽略']
有效目标数： 15


## 3. 合并两种注意力约束

对第 `b` 条样本，第 `i` 个 query 与第 `j` 个 key：

\[
\mathrm{allowed}_{b,i,j}=(j\leq i)\ \land\ (\mathrm{input}_{b,j}\neq\mathrm{PAD})
\]

第一项阻止读取未来，第二项阻止读取补齐 key。形状由 `[B,T]` 扩展成 `[B,1,T,T]`，
其中大小为 1 的头维度会广播给所有头。

| 机制 | 作用位置 | 作用 |
|---|---|---|
| 因果 mask | 注意力分数 | 不让位置 i 读取未来 j |
| 有效 key mask | 注意力分数 | 不读取 PAD 作为 key/value |
| `ignore_index=-100` | 交叉熵标签 | 不让补齐 query 成为训练目标 |

本课 `True` 表示允许，与 SDPA 布尔 mask 的约定一致；不是所有 PyTorch 注意力接口都采用相同约定。[3]

**只支持右 padding，且每条样本第一位有效。** padded query 仍可能产生非零向量，但不参与损失。
不能把 padded query 的整行也直接全部改为 `-inf`，否则普通 softmax 会遇到无有效候选的行。

在严格右 padding 的因果序列中，有效 query 本来就无法看到右侧的 PAD。因此“加 PAD 后有效输出不变”
并不能单独证明 key mask 写对了，下面也直接检查 key mask 的结构。

In [7]:
def attention_allowed(key_valid: torch.Tensor) -> torch.Tensor:
    """[B,T] -> [B,1,T,T]；True=允许读取。只接受非空、右侧补齐。"""
    if (not isinstance(key_valid, torch.Tensor) or key_valid.dtype != torch.bool
            or key_valid.ndim != 2 or min(key_valid.shape) <= 0):
        raise ValueError("key_valid 必须是非空 [B,T] 布尔张量")
    if not bool(key_valid[:, 0].all()):
        raise ValueError("首位置必须有效；不接受全 PAD 或左侧 padding")
    if bool((~key_valid[:, :-1] & key_valid[:, 1:]).any()):
        raise ValueError("只接受连续有效前缀加右 padding，不接受内部空洞")
    t = key_valid.shape[1]
    causal = torch.ones(t, t, dtype=torch.bool, device=key_valid.device).tril()
    # 不屏蔽整行 query：补齐 query 可有数值，但其标签被忽略，避免 softmax 全 -inf。
    return causal[None, None, :, :] & key_valid[:, None, None, :]

In [8]:
key_valid_demo = torch.tensor([[True, True, True], [True, False, False]])
mask_demo = attention_allowed(key_valid_demo)
print("形状：", tuple(mask_demo.shape))
print("完整样本：\n", mask_demo[0, 0].int())
print("仅首位有效：\n", mask_demo[1, 0].int())
assert mask_demo.any(dim=-1).all()
assert not mask_demo[1, :, :, 1:].any()

形状： (2, 1, 3, 3)
完整样本：
 tensor([[1, 0, 0],
        [1, 1, 0],
        [1, 1, 1]], dtype=torch.int32)
仅首位有效：
 tensor([[1, 0, 0],
        [1, 0, 0],
        [1, 0, 0]], dtype=torch.int32)


## 4. 接回多头注意力

这部分沿用第 16 课的数学计算，新增的是传入合并好的 `allowed`：

\[
A=\mathrm{softmax}\left(\mathrm{mask}\left(\frac{QK^\top}{\sqrt{d_{head}}}\right)\right),
\qquad O=AV.
\]

`Q/K/V` 由当前隐藏状态投影得到，形状是 `[B,H,T,d_head]`。各个头合并后做输出投影。
这里直接写矩阵运算，不调用 `nn.Transformer`，方便逐步核对；也没有 FlashAttention 或 KV cache。
注意力 Dropout 使用 `nn.Dropout`，会跟随模型的 train/eval 模式。

In [9]:
class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.n_heads = cfg.n_heads
        self.d_head = cfg.d_model // cfg.n_heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.attention_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        q = self.q_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(b, t, self.n_heads, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        scores = scores.masked_fill(~allowed, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        heads = self.attention_dropout(weights) @ v
        merged = heads.transpose(1, 2).contiguous().reshape(b, t, d)
        return self.out_proj(merged)

## 5. 多个 Block 不是重复使用同一套参数

一个 Block 是：

\[
U=X+\mathrm{MHA}(\mathrm{LN}_1(X)),\qquad
Y=U+\mathrm{FFN}(\mathrm{LN}_2(U)).
\]

这里的 `nn.ModuleList` 注册每一层的子模块，保证 `parameters()`、设备迁移和 `state_dict()` 能找到它们。[1]

正确创建：`[TransformerBlock(cfg) for _ in range(n_layers)]`。
错误创建：先造一个 `block`，再用 `[block] * n_layers`，那会重复引用同一个实例。
共享权重可以是一种有意设计，但不是本课的模型。

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(nn.Linear(cfg.d_model, cfg.ff_dim), nn.GELU(),
                                 nn.Linear(cfg.ff_dim, cfg.d_model))
        self.residual_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, allowed: torch.Tensor) -> torch.Tensor:
        x = x + self.residual_dropout(self.attn(self.norm1(x), allowed))
        x = x + self.residual_dropout(self.ffn(self.norm2(x)))
        return x

## 6. 从隐藏状态到完整语言模型

本课采用下列教学配置：

```text
Token IDs                      [B,T]
   ↓ token embedding + position embedding
隐藏状态                       [B,T,64]
   ↓ Transformer Block 1
   ↓ Transformer Block 2
最终 LayerNorm                 [B,T,64]
   ↓ 独立 Linear 输出头
词表 logits                    [B,T,V]
```

输入 Embedding 是“按编号查向量”，输出头是“按隐藏向量计算所有候选分数”。本课两张参数表彼此独立；
输出头没有偏置，也没有 softmax。模型从随机权重训练，未加载任何预训练模型。

相比原始 GPT 的公开实现，这里保留 decoder-only 自回归思路，但缩小规模，并采用自己明确写出的
字符词表、Pre-LN、独立输出头等教学配置；不能据此声称复现了某个商业模型。[7]

In [11]:
class TinyGPT(nn.Module):
    """输入 [B,T]，输出 [B,T,V] 原始 logits。forward 不计算损失，不移位标签。"""
    def __init__(self, cfg: GPTConfig) -> None:
        super().__init__()
        self.cfg = cfg
        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=PAD_ID)
        self.position_embedding = nn.Embedding(cfg.max_len, cfg.d_model)
        self.embedding_dropout = nn.Dropout(cfg.dropout)
        # 每次创建新的 Block；不要使用 [一个Block] * n_layers。
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        # 本课不共享输出头与输入 Embedding 的权重，方便辨认两个角色。
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.apply(self._init_weights)
        with torch.no_grad():
            self.token_embedding.weight[PAD_ID].zero_()

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        if (not isinstance(input_ids, torch.Tensor) or input_ids.dtype != torch.long
                or input_ids.ndim != 2 or min(input_ids.shape) <= 0):
            raise ValueError("input_ids 必须是非空 [B,T] torch.long 张量")
        if input_ids.device != self.token_embedding.weight.device:
            raise ValueError("输入与模型必须位于同一设备")
        if bool(((input_ids < 0) | (input_ids >= self.cfg.vocab_size)).any()):
            raise ValueError("输入包含词表范围之外的编号")
        t = input_ids.shape[1]
        if t > self.cfg.max_len:
            raise ValueError(f"输入长度 {t} 超过上下文 {self.cfg.max_len}；本课不自动截断")
        allowed = attention_allowed(input_ids != PAD_ID)
        positions = torch.arange(t, device=input_ids.device)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)[None, :, :]
        x = self.embedding_dropout(x)
        for block in self.blocks:
            x = block(x, allowed)
        return self.lm_head(self.final_norm(x))


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def parameter_formula(cfg: GPTConfig) -> int:
    d, f = cfg.d_model, cfg.ff_dim
    # 每块：4 个 D→D + 两个 LayerNorm + D→F→D。
    per_block = 4 * (d*d + d) + 4*d + (d*f + f) + (f*d + d)
    return 2 * cfg.vocab_size*d + cfg.max_len*d + cfg.n_layers*per_block + 2*d

In [12]:
set_seed(17)
preview_cfg = GPTConfig(vocab_size=len(id_to_token))
preview_model = TinyGPT(preview_cfg).to(DEVICE).eval()
preview_ids = preview_batch["input_ids"].to(DEVICE)
with torch.no_grad():
    positions = torch.arange(preview_ids.shape[1], device=DEVICE)
    hidden = preview_model.token_embedding(preview_ids) + preview_model.position_embedding(positions)[None]
    print("Embedding 后：", tuple(hidden.shape))
    allowed = attention_allowed(preview_ids != PAD_ID)
    for i, block in enumerate(preview_model.blocks, start=1):
        hidden = block(hidden, allowed)
        print(f"Block {i} 后：", tuple(hidden.shape))
    manual_logits = preview_model.lm_head(preview_model.final_norm(hidden))
    expected_logits = preview_model(preview_ids)
torch.testing.assert_close(manual_logits, expected_logits)
print("logits：", tuple(expected_logits.shape))
print("参数量（遍历 / 公式）：", count_parameters(preview_model), parameter_formula(preview_cfg))
print("两层是否同一对象：", preview_model.blocks[0] is preview_model.blocks[1])

Embedding 后： (2, 11, 64)
Block 1 后： (2, 11, 64)
Block 2 后： (2, 11, 64)
logits： (2, 11, 41)
参数量（遍历 / 公式）： 109440 109440
两层是否同一对象： False


### 参数量可以自己算

默认训练词表有 41 个 token。设 `D=64, F=256, L=2, T_max=64`：

| 部件 | 参数量 |
|---|---:|
| Token Embedding | `41×64 = 2,624` |
| Position Embedding | `64×64 = 4,096` |
| 每个 Block | `4(D²+D) + 4D + (DF+F) + (FD+D) = 49,984` |
| 两个 Block | `99,968` |
| 最终 LayerNorm | `2×64 = 128` |
| 无偏置输出头 | `64×41 = 2,624` |
| **总计** | **109,440** |

这只是参数数量，不是训练峰值显存；激活、梯度、优化器状态也要占用内存。
如果更换语料或配置，实际参数量会变化，以程序输出为准。

## 7. 先做机制检查，再训练

一个语言模型 loss 能下降，并不能证明没有标签泄漏。先检验整个两层模型：

- 改变最后一个输入 token，前面的 logits 应保持不变。
- 只输入前缀，所得 logits 应与整句中对应前缀一致。
- 改变前文，最后位置的输出应有机会改变。

测试时要使用 `eval()` 关闭随机 Dropout，否则两次前向的随机差异会干扰判断。

In [13]:
probe_ids = torch.tensor([[BOS_ID, token_to_id["我"], token_to_id["今"], token_to_id["天"]]], device=DEVICE)
future_changed = probe_ids.clone()
future_changed[0, -1] = token_to_id["明"]
context_changed = probe_ids.clone()
context_changed[0, 1] = token_to_id["你"]
with torch.no_grad():
    original = preview_model(probe_ids)
    after_future = preview_model(future_changed)
    after_context = preview_model(context_changed)
    prefix_only = preview_model(probe_ids[:, :3])
torch.testing.assert_close(original[:, :-1], after_future[:, :-1])
torch.testing.assert_close(original[:, :3], prefix_only)
print("改变未来，前缀最大差异：", (original[:, :-1]-after_future[:, :-1]).abs().max().item())
print("改变前文，末位置最大差异：", (original[:, -1]-after_context[:, -1]).abs().max().item())

改变未来，前缀最大差异： 0.0
改变前文，末位置最大差异： 0.12209342420101166


## 8. 交叉熵：预测全部有效位置

定义有效位置集合 \(\mathcal V\)，训练目标为：

\[
L=-\frac{1}{|\mathcal V|}\sum_{(b,t)\in\mathcal V}
\log p_\theta(y_{b,t}\mid x_{b,0:t}).
\]

`logits` 是 `[B,T,V]`，目标是 `[B,T]` 的 `long` 张量。展平前两轴后直接送入交叉熵，
**不先 softmax，也不再次移位**。`ignore_index=-100` 跳过补齐目标，EOS 是有效目标。[2]

验证时把每批的损失总和与有效 token 数分别累加，最后相除。不能直接平均不同大小批次的 loss。
`eval()` 与 `no_grad()` 分别控制模块行为和梯度记录；评估结束后恢复调用前整体模式。[8]

In [14]:
def loss_parts(logits: torch.Tensor, targets: torch.Tensor) -> tuple[torch.Tensor, int]:
    if (logits.ndim != 3 or targets.shape != logits.shape[:2]
            or targets.dtype != torch.long or logits.device != targets.device):
        raise ValueError("需要同设备 logits=[B,T,V]、long targets=[B,T]")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits 包含非有限值")
    valid = targets != IGNORE_INDEX
    count = int(valid.sum().item())
    if count == 0:
        raise ValueError("本批没有有效目标，不能计算平均损失")
    if bool(((targets[valid] < 0) | (targets[valid] >= logits.shape[-1])).any()):
        raise ValueError("有效目标编号越界")
    total = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1),
                            ignore_index=IGNORE_INDEX, reduction="sum")
    return total, count


def finish_metrics(total_loss: float, total_correct: int, total_tokens: int) -> dict:
    if total_tokens <= 0:
        raise ValueError("没有可评估的有效 token")
    mean = total_loss / total_tokens
    if not math.isfinite(mean):
        raise ValueError("平均损失不是有限数")
    # JSON 不能保存 Inf；异常大损失时不报告 PPL 数值。
    ppl = math.exp(mean) if mean < 700 else None
    return {"loss": mean, "perplexity": ppl, "accuracy": total_correct / total_tokens,
            "tokens": total_tokens}


def evaluate(model: TinyGPT, loader: DataLoader, device: torch.device) -> dict:
    was_training = model.training
    total_loss, total_correct, total_tokens = 0.0, 0, 0
    try:
        model.eval()
        with torch.no_grad():
            for batch in loader:
                inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
                logits = model(inputs)
                loss_sum, count = loss_parts(logits, targets)
                valid = targets != IGNORE_INDEX
                total_loss += loss_sum.item()
                total_correct += int(((logits.argmax(-1) == targets) & valid).sum().item())
                total_tokens += count
    finally:
        model.train(was_training)
    return finish_metrics(total_loss, total_correct, total_tokens)

In [15]:
loss_sum, valid_tokens = loss_parts(expected_logits, preview_batch["targets"].to(DEVICE))
print("损失总和：", loss_sum.item())
print("有效目标数：", valid_tokens)
print("平均交叉熵：", (loss_sum / valid_tokens).item())
modified_logits = expected_logits.clone()
modified_logits[preview_batch["targets"].to(DEVICE) == IGNORE_INDEX] = 500.0
modified_loss, _ = loss_parts(modified_logits, preview_batch["targets"].to(DEVICE))
torch.testing.assert_close(loss_sum, modified_loss)
print("只改补齐位置 logits，损失保持不变。")

损失总和： 55.47297286987305
有效目标数： 15
平均交叉熵： 3.698198080062866
只改补齐位置 logits，损失保持不变。


### 困惑度 PPL：平均负对数概率的指数

使用自然对数的平均交叉熵时，\(\mathrm{PPL}=e^L\)。如果 41 个候选始终等概率，损失为 \(\log 41\)，
PPL 就是 41。本课包含 EOS 目标，不包含 padding 目标。

不要把 PPL 理解为“模型 IQ”或真实候选数量。分词方式、评估语料、上下文与特殊 token 计分规则
都会影响它，字符模型的 PPL 不应直接与 BPE 模型 PPL 对比。[10]

In [16]:
uniform_loss = math.log(len(id_to_token))
print("均匀分布交叉熵：", uniform_loss)
print("均匀分布 PPL：", math.exp(uniform_loss))

均匀分布交叉熵： 3.713572066704308
均匀分布 PPL： 41.00000000000001


## 9. 完整训练步：清零 → 前向 → 反向 → 裁剪 → 更新

本课使用 AdamW，学习率 `0.003`，权重衰减 `0.01`，最大梯度范数 `1.0`。这是固定教学配方，
不是通用最优参数，没有学习率预热或调度器，也没有参数分组。[4]

```python
optimizer.zero_grad(set_to_none=True)
logits = model(input_ids)
loss_sum, token_count = loss_parts(logits, targets)
(loss_sum / token_count).backward()
clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
optimizer.step()
```

裁剪操作修改的是梯度，发生在 `backward()` 后和 `step()` 前；返回的范数是裁剪前的总范数。
它不能补救标签错误、泄漏或不合理的输入，遇到非有限梯度本课直接报告错误。[5]

一轮中的在线训练 loss 来自不断变化的权重；为了比较轮次，另用当轮结束后的固定模型计算 train/validation 指标。
保存最佳权重要复制张量；只写 `best_state = model.state_dict()` 可能保留继续变化的引用。[6]

In [17]:
def train_epoch(model: TinyGPT, loader: DataLoader, optimizer: torch.optim.Optimizer,
                device: torch.device, max_grad_norm: float = 1.0) -> dict:
    model.train()
    total_loss, total_correct, total_tokens, steps = 0.0, 0, 0, 0
    for batch in loader:
        inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(inputs)
        loss_sum, count = loss_parts(logits, targets)
        (loss_sum / count).backward()
        # 裁剪发生在 backward 之后、step 之前；不是直接截断参数。
        nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm, error_if_nonfinite=True)
        optimizer.step()
        valid = targets != IGNORE_INDEX
        total_loss += loss_sum.detach().item()
        total_correct += int(((logits.detach().argmax(-1) == targets) & valid).sum().item())
        total_tokens += count
        steps += 1
    result = finish_metrics(total_loss, total_correct, total_tokens)
    result["steps"] = steps
    return result  # 在线指标来自不断改变的参数；选模型用下面的固定参数评估。


def snapshot_state(model: nn.Module) -> dict[str, torch.Tensor]:
    # clone 防止“最佳参数”引用继续被后续训练修改。
    return {name: tensor.detach().cpu().clone() for name, tensor in model.state_dict().items()}

## 10. 生成：用自己的输出继续下一步

训练时用真实前缀；生成时用当前已有前缀，再追加自己选出的 token：

```text
<bos> 我 今 天 → 预测“正”
<bos> 我 今 天 正 → 预测“在”
……
```

只读取 `logits[0, -1, :]`。贪心取最大值；采样使用 `softmax(logits / temperature)` 后抽样。
生成不做反向传播、不更新权重。BOS/PAD 被禁止生成，EOS 终止生成，UNK 若被采到会明确显示。

本课对未知提示字符报错，避免把用户输入悄悄替换成 UNK。训练/验证编码仍可使用 UNK，这是两种明确不同的接口约定。

**没有 KV cache：每生成一字都重算整个前缀。** 超长提示会报错；循环达到上下文边界时返回 `context_limit`，
不会自动裁掉旧 token。长度恰为 `max_len` 的输入仍可预测最后一个新 token，因此返回编号可能有 `max_len+1` 项，
但不会再次把这个更长的序列送入模型。

In [18]:
def generate(model: TinyGPT, id_to_token: list[str], prompt: str = "", *,
             max_new_tokens: int = 24, sample: bool = False,
             temperature: float = 0.8, seed: int = 17) -> dict:
    validate_vocab(id_to_token)
    if len(id_to_token) != model.cfg.vocab_size:
        raise ValueError("模型与词表大小不一致")
    if not isinstance(prompt, str):
        raise ValueError("prompt 必须是字符串")
    if type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError("max_new_tokens 必须是非负整数")
    if type(sample) is not bool:
        raise ValueError("sample 必须是 bool")
    if (isinstance(temperature, bool) or not isinstance(temperature, (int, float))
            or not math.isfinite(temperature) or temperature <= 0):
        raise ValueError("temperature 必须是有限正数；贪心请用 sample=False")
    token_to_id = {t: i for i, t in enumerate(id_to_token)}
    unknown = sorted(set(prompt) - set(token_to_id))
    if unknown:
        raise ValueError(f"生成提示含词表外字符：{unknown}；本课拒绝静默替换提示")
    ids = [BOS_ID] + encode(prompt, token_to_id)
    if len(ids) > model.cfg.max_len:
        raise ValueError("提示加 BOS 已超过上下文长度")
    device = next(model.parameters()).device
    rng = torch.Generator(device=device).manual_seed(seed)
    generated, reason = [], "max_new_tokens"
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                if len(ids) > model.cfg.max_len:
                    reason = "context_limit"
                    break
                input_ids = torch.tensor([ids], dtype=torch.long, device=device)
                next_logits = model(input_ids)[0, -1, :].clone()
                # 生成时不允许输出起始符或补齐符；这是解码规则，不是训练时改词表。
                next_logits[[BOS_ID, PAD_ID]] = float("-inf")
                if sample:
                    probabilities = torch.softmax(next_logits / temperature, dim=-1)
                    next_id = int(torch.multinomial(probabilities, 1, generator=rng).item())
                else:
                    next_id = int(next_logits.argmax().item())
                ids.append(next_id)
                generated.append(next_id)
                if next_id == EOS_ID:
                    reason = "eos"
                    break
    finally:
        model.train(was_training)
    visible = []
    for index in generated:
        if index == EOS_ID:
            break
        visible.append(id_to_token[index])  # <unk> 如被采到则明确显示。
    return {"prompt": prompt, "completion": "".join(visible), "text": prompt + "".join(visible),
            "generated_ids": generated, "stop_reason": reason,
            "sample": sample, "temperature": temperature if sample else None, "seed": seed}

## 11. 保存哪些东西，才能正确恢复？

保存 `state_dict + 模型配置 + 词表顺序 + 最佳轮次`。词表编号一旦改变，相同权重也会被错误解释。
重新构造相同模型，再严格加载参数，验证 logits 与生成结果。[6]

本课检查点用于推理复现，**没有优化器、数据遍历进度、随机数状态，不能精确接着中断的训练继续**。
只加载自己生成或信任的文件；`weights_only=True` 并不等于任意外部文件都安全。

In [19]:
def save_checkpoint(path: Path, model: TinyGPT, id_to_token: list[str], best_epoch: int) -> None:
    validate_vocab(id_to_token)
    if len(id_to_token) != model.cfg.vocab_size:
        raise ValueError("模型与词表大小不一致")
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    torch.save({"format_version": 1, "config": asdict(model.cfg), "vocab": id_to_token,
                "model_state": snapshot_state(model), "best_epoch": best_epoch,
                "purpose": "inference_only_not_exact_training_resume"}, path)


def load_checkpoint(path: Path, device: torch.device) -> tuple[TinyGPT, list[str], dict]:
    # 只加载自己生成/信任的文件；weights_only 也不是任意文件安全性的保证。
    payload = torch.load(path, map_location="cpu", weights_only=True)
    if not isinstance(payload, dict) or payload.get("format_version") != 1:
        raise ValueError("不支持的检查点格式")
    vocab = payload["vocab"]
    validate_vocab(vocab)
    cfg = GPTConfig(**payload["config"])
    if len(vocab) != cfg.vocab_size:
        raise ValueError("检查点词表与配置不一致")
    # 构造随机初始化模型不消耗调用者的 CPU 随机状态。
    with torch.random.fork_rng(devices=[]):
        model = TinyGPT(cfg)
    model.load_state_dict(payload["model_state"], strict=True)
    model.to(device).eval()
    return model, vocab, {"best_epoch": payload["best_epoch"], "purpose": payload["purpose"]}

## 12. 运行端到端实验

`run_experiment()` 把已讲解的步骤按顺序连接起来：

```text
读取/切分 → 构造 DataLoader → 随机初始化 → 训练前评估
→ 每轮训练与验证 → 复制最优权重 → 保存/重载 → 生成样例 → 保存指标
```

验证集用于选轮次；本课没有最终独立测试集。**最佳验证成绩不是未参与选择的测试成绩。**
下面默认 40 轮，batch=16，训练集 77 条，所以每轮 5 次参数更新，总共 200 次。

默认输入 `lesson_17_data/corpus.txt` 缺失时才创建；已有内容保留。
输出 `lesson_17_outputs` 下的同名文件会覆盖；改变输出目录可保留不同实验。
程序会重设训练随机种子，所以之前运行多少个示例不应改变同环境下这次默认训练。

In [20]:
def run_experiment(text_path: Path, output_dir: Path, *, device: torch.device,
                   epochs: int = 40, batch_size: int = 16, seed: int = 17,
                   max_len: int = 64, d_model: int = 64, n_heads: int = 4,
                   n_layers: int = 2, prompts: list[str] | None = None) -> dict:
    positive_int(epochs, "epochs")
    positive_int(batch_size, "batch_size")
    text_path, output_dir = Path(text_path), Path(output_dir)
    output_names = ["split.json", "metrics.json", "samples.json", "best_model.pt"]
    if any(text_path.resolve() == (output_dir / name).resolve() for name in output_names):
        raise ValueError("输入文件不能与生成文件同路径")
    raw = text_path.read_text(encoding="utf-8")
    texts = clean_texts(raw.splitlines())
    train_texts, val_texts = split_texts(texts, seed)
    token_to_id, vocab = build_vocab(train_texts)
    cfg = GPTConfig(len(vocab), max_len=max_len, d_model=d_model, n_heads=n_heads,
                    n_layers=n_layers, ff_dim=4*d_model, dropout=0.1)
    train_data = CharDataset(train_texts, token_to_id, max_len)
    val_data = CharDataset(val_texts, token_to_id, max_len)
    train_loader = make_loader(train_data, batch_size, True, seed)
    train_eval_loader = make_loader(train_data, batch_size)
    val_loader = make_loader(val_data, batch_size)
    set_seed(seed)
    model = TinyGPT(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
    before = {"train": evaluate(model, train_eval_loader, device),
              "validation": evaluate(model, val_loader, device)}
    print(f"设备={device} | 训练={len(train_data)} 条 | 验证={len(val_data)} 条 | 词表={len(vocab)}")
    print(f"参数={count_parameters(model):,} | {n_layers} 层 | D={d_model} | 头数={n_heads}")
    best_loss, best_epoch, best_state = before["validation"]["loss"], 0, snapshot_state(model)
    history, total_steps = [], 0
    for epoch in range(1, epochs + 1):
        online = train_epoch(model, train_loader, optimizer, device)
        total_steps += online["steps"]
        train_metrics = evaluate(model, train_eval_loader, device)
        val_metrics = evaluate(model, val_loader, device)
        history.append({"epoch": epoch, "online_train": online, "train": train_metrics,
                        "validation": val_metrics})
        if val_metrics["loss"] < best_loss:
            best_loss, best_epoch, best_state = val_metrics["loss"], epoch, snapshot_state(model)
        if epoch == 1 or epoch % 10 == 0 or epoch == epochs:
            print(f"epoch={epoch:02d} | train={train_metrics['loss']:.4f} | val={val_metrics['loss']:.4f}")
    last = {"train": history[-1]["train"], "validation": history[-1]["validation"]}
    model.load_state_dict(best_state)
    model.eval()
    best = {"epoch": best_epoch, "train": evaluate(model, train_eval_loader, device),
            "validation": evaluate(model, val_loader, device)}
    output_dir.mkdir(parents=True, exist_ok=True)
    save_checkpoint(output_dir / "best_model.pt", model, vocab, best_epoch)
    loaded, loaded_vocab, _ = load_checkpoint(output_dir / "best_model.pt", device)
    probe = next(iter(val_loader))["input_ids"].to(device)
    with torch.no_grad():
        reload_diff = float((model(probe) - loaded(probe)).abs().max().item())
    torch.testing.assert_close(model(probe), loaded(probe), rtol=0, atol=0)
    if prompts is None:
        candidates = ["我今天", "小明明天", "老师今天"]
        prompts = [p for p in candidates if all(c in token_to_id for c in p)]
        if not prompts:
            prompts = [train_texts[0][:min(2, max_len - 1)]]
    samples = []
    for i, prompt in enumerate(prompts):
        for sample in (False, True):
            result = generate(loaded, loaded_vocab, prompt, sample=sample, seed=seed+i)
            samples.append(result)
            print(("采样" if sample else "贪心") + f"：{result['text']}  [{result['stop_reason']}]")
    val_char_count = sum(len(t) for t in val_texts)
    val_unknown = sum(c not in token_to_id for t in val_texts for c in t)
    report = {"lesson": 17, "torch_version": str(torch.__version__), "device": str(device),
              "seed": seed, "config": asdict(cfg), "parameters": count_parameters(model),
              "epochs": epochs, "batch_size": batch_size, "optimizer_steps": total_steps,
              "training_recipe": {"optimizer": "AdamW", "lr": 0.003, "weight_decay": 0.01,
                                  "clip_grad_norm": 1.0, "scheduler": None},
              "input_path": str(text_path.resolve()), "corpus_sha256": hashlib.sha256(raw.encode()).hexdigest(),
              "data": {"unique_lines": len(texts), "train_lines": len(train_texts),
                       "validation_lines": len(val_texts), "validation_unknown_characters": val_unknown,
                       "validation_characters": val_char_count},
              "before": before, "last": last, "best": best, "history": history,
              "reload_max_abs_difference": reload_diff,
              "limitations": "手工模板语料；验证参与选轮次，无独立测试；非对话助手；无 KV cache；非精确续训检查点"}
    for name, payload in [("split.json", {"train": train_texts, "validation": val_texts, "vocab": vocab}),
                          ("metrics.json", report), ("samples.json", samples)]:
        (output_dir / name).write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False),
                                      encoding="utf-8")
    print(f"最佳轮次={best_epoch} | 验证损失={best_loss:.6f} | 重载差异={reload_diff}")
    print("结果目录：", output_dir.resolve())
    return report

In [21]:
TEXT_PATH = ensure_demo_file(Path("lesson_17_data/corpus.txt"))
OUTPUT_DIR = Path("lesson_17_outputs")
report = run_experiment(TEXT_PATH, OUTPUT_DIR, device=DEVICE, epochs=40,
                        batch_size=16, seed=17)

设备=cpu | 训练=77 条 | 验证=19 条 | 词表=41
参数=109,440 | 2 层 | D=64 | 头数=4
epoch=01 | train=2.5748 | val=2.6290


epoch=10 | train=0.4321 | val=0.4717


epoch=20 | train=0.3962 | val=0.4369


epoch=30 | train=0.3911 | val=0.4255


epoch=40 | train=0.3896 | val=0.4504
贪心：我今天正在学习数学。  [eos]
采样：我今天正在学习数学。  [eos]
贪心：小明明天正在学习数学。  [eos]
采样：小明明天正在训练模型。  [eos]
贪心：老师今天正在学习数学。  [eos]
采样：老师今天正在整理笔记。  [eos]
最佳轮次=29 | 验证损失=0.425096 | 重载差异=0.0
结果目录： /mnt/data/_lesson17_work/notebook_run/lesson_17_outputs


## 13. 读懂结果，而不是只看生成得像不像

下面同时展示训练前、最佳验证轮次和最后一轮的指标。所有 train 指标都是用该状态的固定模型在 eval 模式下重新计算的，
不是在线训练 loss。

即使语句流畅，也可能主要复用了模板。同一个前缀存在多个合法后续时，loss 不必为零；
token accuracy 也不是“整句话正确率”或“通用语言能力”。

In [22]:
stages = [("训练前", report["before"]),
          (f"最佳轮次 {report['best']['epoch']}", report["best"]),
          (f"最后轮次 {report['epochs']}", report["last"])]
print(f"{'阶段':<12} {'训练 loss':>12} {'验证 loss':>12} {'验证 PPL':>12}")
for label, value in stages:
    print(f"{label:<12} {value['train']['loss']:>12.6f} {value['validation']['loss']:>12.6f} {value['validation']['perplexity']:>12.6f}")
print("验证未知字符：", report["data"]["validation_unknown_characters"])
print("验证有效目标：", report["best"]["validation"]["tokens"])
print("重载最大差异：", report["reload_max_abs_difference"])

阶段                训练 loss      验证 loss       验证 PPL
训练前              3.708465     3.718230    41.191438
最佳轮次 29          0.393359     0.425096     1.529737
最后轮次 40          0.389648     0.450399     1.568938
验证未知字符： 0
验证有效目标： 222
重载最大差异： 0.0


### 单独加载，不重新训练

下面使用已保存的权重进行生成。修改 `prompt` 或 `seed` 可以改变输入或采样随机性；这不会修改参数。
如果换成自己的语料，要用该词表中存在的字符作为提示。

In [23]:
loaded_model, loaded_vocab, checkpoint_info = load_checkpoint(OUTPUT_DIR / "best_model.pt", DEVICE)
loaded_mapping = {t: i for i, t in enumerate(loaded_vocab)}
my_prompt = "小明明天" if all(c in loaded_mapping for c in "小明明天") else ""
weights_before_generation = snapshot_state(loaded_model)
print("贪心：", generate(loaded_model, loaded_vocab, my_prompt, sample=False)["text"])
for seed in (1, 2, 3):
    result = generate(loaded_model, loaded_vocab, my_prompt, sample=True,
                      temperature=0.8, seed=seed)
    print(f"seed={seed}：{result['text']} | {result['stop_reason']}")
assert all(torch.equal(v.cpu(), weights_before_generation[k]) for k, v in loaded_model.state_dict().items())
print("生成没有修改参数。")

贪心： 小明明天正在学习数学。
seed=1：小明明天正在整理笔记。 | eos
seed=2：小明明天正在训练模型。 | eos
seed=3：小明明天正在检查数据。 | eos
生成没有修改参数。


## 14. 固定自动检查

测试覆盖数据移位、mask 结构、模型注册、独立层参数、因果性、梯度、padding、
验证汇总、保存重载、生成停止条件，以及单样本拟合。

检查只验证列出的约定，不能证明任意输入都没有缺陷。这里的实现故意拒绝左 padding、内部空洞和超长输入。
下一个代码单元较长，已默认折叠；可以展开查看每条检查的原因。临时检查点会在检查结束后删除。

In [24]:
class Checks:
    def __init__(self) -> None:
        self.count = 0

    def ok(self, condition: bool, message: str) -> None:
        if not bool(condition):
            raise AssertionError(message)
        self.count += 1

    def close(self, actual: torch.Tensor, expected: torch.Tensor, message: str,
              atol: float = 1e-6, rtol: float = 1e-5) -> None:
        torch.testing.assert_close(actual, expected, atol=atol, rtol=rtol, msg=message)
        self.count += 1

    def raises(self, function, message: str, kind=ValueError) -> None:
        try:
            function()
        except kind:
            self.count += 1
        else:
            raise AssertionError(message)


def run_checks(device: torch.device = torch.device("cpu")) -> int:
    c = Checks()
    set_seed(1701)
    mapping, vocab = build_vocab(["我爱学习。", "你爱模型。"])
    cfg = GPTConfig(len(vocab), max_len=16, d_model=16, n_heads=4, n_layers=2, ff_dim=32, dropout=0.0)
    model = TinyGPT(cfg).to(device).eval()
    data = CharDataset(["我爱学习。", "我", "", "你爱模型。"], mapping, cfg.max_len)
    batch = collate_batch([data[i] for i in range(len(data))])
    inputs, targets = batch["input_ids"].to(device), batch["targets"].to(device)

    # 数据与词表
    c.ok(len(demo_lines()) == len(set(demo_lines())) == 96, "默认模板应为 96 条不同文本")
    c.ok(clean_texts([" 我 ", "我", "", "  ", "你"]) == ["我", "你"], "去重/去空失败")
    train, val = split_texts(demo_lines())
    c.ok(len(train) == 77 and len(val) == 19 and not set(train)&set(val), "按完整文本切分失败")
    c.ok((train, val) == split_texts(demo_lines()), "固定切分种子失效")
    c.ok(vocab[:4] == SPECIAL_TOKENS, "特殊 token 顺序不一致")
    c.ok(encode("火", mapping) == [UNK_ID] and "火" not in mapping, "词表不应在编码时扩充")
    c.ok(encode("", mapping) == [], "空字符串编码错误")
    x, y = data[0]
    c.ok(x.tolist() == [BOS_ID]+encode("我爱学习。", mapping), "输入错位")
    c.ok(y.tolist() == encode("我爱学习。", mapping)+[EOS_ID], "标签未移位或重复移位")
    c.ok(data[2][0].tolist() == [BOS_ID] and data[2][1].tolist() == [EOS_ID], "空文本边界错误")
    c.ok(inputs.shape == targets.shape == (4, 6), "补齐形状错误")
    c.ok(batch["input_ids"][1].tolist() == [BOS_ID, mapping["我"]]+[PAD_ID]*4, "输入应右补 PAD")
    c.ok(batch["targets"][1].tolist() == [mapping["我"], EOS_ID]+[IGNORE_INDEX]*4, "标签必须补 -100")
    c.ok(sum(b["input_ids"].shape[0] for b in make_loader(data, 3)) == len(data), "不能丢最后一批")
    c.raises(lambda: CharDataset(["我"*20], mapping, 16), "长文本必须明确拒绝")
    c.raises(lambda: split_texts(["同句"]*5), "重复文本应在切分前处理")
    c.raises(lambda: collate_batch([]), "空 batch 应拒绝")

    # 掩码与模型结构
    valid = torch.tensor([[True, True, True], [True, False, False]], device=device)
    allowed = attention_allowed(valid)
    c.ok(allowed.shape == (2, 1, 3, 3), "mask 形状错误")
    c.ok(torch.equal(allowed[0, 0], torch.ones(3, 3, dtype=torch.bool, device=device).tril()), "因果方向错误")
    c.ok(not bool(allowed[1, :, :, 1:].any()), "PAD key 未屏蔽")
    c.ok(bool(allowed.any(-1).all()), "不能制造整行全被屏蔽")
    for invalid in ([[False, False]], [[False, True]], [[True, False, True]]):
        c.raises(lambda v=invalid: attention_allowed(torch.tensor(v, device=device)), "非法补齐必须拒绝")
    c.ok(isinstance(model.blocks, nn.ModuleList), "Block 必须注册为子模块")
    c.ok(model.blocks[0].attn.q_proj.weight.data_ptr() != model.blocks[1].attn.q_proj.weight.data_ptr(), "不能意外重复同一 Block")
    c.ok(model.lm_head.weight.data_ptr() != model.token_embedding.weight.data_ptr(), "本课输出头不共享参数")
    c.ok(count_parameters(model) == parameter_formula(cfg), "参数计数公式不匹配")
    c.ok(any(name.startswith("blocks.1.") for name in model.state_dict()), "第二层参数未注册")
    c.ok(torch.equal(model.token_embedding.weight[PAD_ID], torch.zeros(cfg.d_model, device=device)), "PAD 行初始应为零")
    c.raises(lambda: GPTConfig(len(vocab), d_model=15, n_heads=4), "头维度不能整除应报错")
    c.raises(lambda: GPTConfig(len(vocab), n_layers=0), "层数必须为正")
    c.raises(lambda: GPTConfig(len(vocab), dropout=1), "dropout=1 应拒绝")
    for invalid in (torch.ones(1, 2, device=device), torch.tensor([1, 2], device=device),
                    torch.tensor([[-1]], device=device), torch.tensor([[len(vocab)]], device=device),
                    torch.empty(1, 0, dtype=torch.long, device=device),
                    torch.ones(1, 17, dtype=torch.long, device=device)):
        c.raises(lambda v=invalid: model(v), "非法模型输入未拒绝")

    logits = model(inputs)
    c.ok(logits.shape == (*inputs.shape, len(vocab)), "必须输出每个位置的词表 logits")
    c.ok(bool(torch.isfinite(logits).all()), "补齐 query 不能产生 NaN")
    # 整个两层模型因果性；不能只测单层注意力。
    with torch.no_grad():
        changed = inputs.clone()
        changed[0, -1] = mapping["你"]
        c.close(model(changed)[0, :-1], logits[0, :-1], "未来改变影响了前缀")
        prefix = inputs[0:1, :3]
        c.close(model(prefix)[0], logits[0, :3], "只计算前缀应与整句的前缀一致")
        c.close(model(inputs[1:2, :2])[0], logits[1, :2], "右 padding 改变有效位置")
        changed_prefix = prefix.clone()
        changed_prefix[0, 1] = mapping["你"]
        c.ok((model(changed_prefix)[0, -1]-model(prefix)[0, -1]).abs().max() > 1e-7, "模型未利用前文")
    # 对投影前隐藏向量求梯度：前缀损失不能回传到未来位置。
    h = torch.randn(1, 4, cfg.d_model, device=device, requires_grad=True)
    mask = attention_allowed(torch.ones(1, 4, dtype=torch.bool, device=device))
    z = h
    for block in model.blocks:
        z = block(z, mask)
    z[:, :2].square().sum().backward()
    c.ok(torch.equal(h.grad[:, 2:], torch.zeros_like(h.grad[:, 2:])), "未来输入接收到前缀损失梯度")
    c.ok(h.grad[:, :2].abs().sum() > 0, "前缀应接收到梯度")

    # 损失与参数更新
    model.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss_sum, n = loss_parts(logits, targets)
    valid_targets = targets != IGNORE_INDEX
    c.ok(n == sum(len(data[i][1]) for i in range(len(data))), "有效 token 计数错误")
    reference = F.cross_entropy(logits[valid_targets], targets[valid_targets], reduction="sum")
    c.close(loss_sum, reference, "忽略位置的损失错误")
    perturbed = logits.detach().clone()
    perturbed[~valid_targets] = torch.randn_like(perturbed[~valid_targets]) * 100
    c.close(loss_parts(perturbed, targets)[0], loss_sum.detach(), "补齐位置影响损失")
    c.close(loss_sum/n, F.cross_entropy(logits.reshape(-1, len(vocab)), targets.reshape(-1),
                                      ignore_index=IGNORE_INDEX), "平均损失错误")
    c.raises(lambda: loss_parts(logits, torch.full_like(targets, IGNORE_INDEX)), "零有效 token 应报错")
    c.raises(lambda: loss_parts(logits, targets.float()), "标签必须为 long")
    (loss_sum/n).backward()
    c.ok(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()), "所有可训练参数应接收有限梯度")
    c.ok(all(block.attn.q_proj.weight.grad.abs().sum() > 0 for block in model.blocks), "两层注意力都应学习")
    c.ok(torch.equal(model.token_embedding.weight.grad[PAD_ID], torch.zeros(cfg.d_model, device=device)), "PAD embedding 不应接收查表梯度")
    saved = snapshot_state(model)
    old_head = model.lm_head.weight.detach().clone()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.003)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    c.ok(not torch.equal(model.lm_head.weight, old_head), "优化器应更新参数")
    c.close(saved["lm_head.weight"], old_head.cpu(), "快照不能随训练改变", atol=0, rtol=0)
    model.zero_grad(set_to_none=True)

    # 评估对 batch 划分不敏感，也不更新模型
    saved = snapshot_state(model)
    model.train()
    m1 = evaluate(model, make_loader(data, 1), device)
    m3 = evaluate(model, make_loader(data, 3), device)
    c.ok(abs(m1["loss"]-m3["loss"]) < 1e-5, "评估必须按有效 token 加权汇总")
    c.ok(m1["tokens"] == m3["tokens"] and m1["accuracy"] == m3["accuracy"], "评估有效 token 与正确数不一致")
    c.ok(model.training, "评估后应恢复整体训练模式")
    c.ok(all(torch.equal(p.cpu(), saved[k]) for k, p in model.state_dict().items()), "评估不能更新参数")
    c.ok(all(p.grad is None for p in model.parameters()), "评估不应产生梯度")
    c.ok(abs(m1["perplexity"] - math.exp(m1["loss"])) < 1e-10, "PPL 应等于 exp(平均交叉熵)")

    # 生成：仅使用末位置、有限步数、模式恢复、EOS/上下文/采样
    before_gen = snapshot_state(model)
    a = generate(model, vocab, "我", max_new_tokens=3)
    c.ok(model.training, "生成应恢复调用前模式")
    c.ok(len(a["generated_ids"]) <= 3, "新增 token 超预算")
    c.ok(BOS_ID not in a["generated_ids"] and PAD_ID not in a["generated_ids"], "不应生成 BOS/PAD")
    c.ok(all(torch.equal(p.cpu(), before_gen[k]) for k, p in model.state_dict().items()), "生成不应更新权重")
    c.ok(generate(model, vocab, "我", max_new_tokens=0)["text"] == "我", "零步生成应保留提示")
    c.raises(lambda: generate(model, vocab, "火"), "未知提示字符应明确报错")
    c.raises(lambda: generate(model, vocab, "我"*16), "超长提示应拒绝")
    c.raises(lambda: generate(model, vocab, "我", temperature=0), "温度零应拒绝")
    c.raises(lambda: generate(model, vocab, "我", max_new_tokens=-1), "负步数应拒绝")
    # 可预测的假模型，仅用于检查生成循环；返回每个位置不同的最优类别。
    class Stub(nn.Module):
        def __init__(self, last_id: int, max_len: int = 4):
            super().__init__()
            self.cfg = GPTConfig(len(vocab), max_len=max_len, d_model=8, n_heads=2, n_layers=1, ff_dim=16)
            self.anchor = nn.Parameter(torch.zeros(1, device=device))
            self.last_id = last_id
        def forward(self, ids):
            out = torch.full((*ids.shape, len(vocab)), -20.0, device=ids.device)
            out[:, :, mapping["你"]] = 20.0
            out[:, -1, :] = -20.0
            out[:, -1, self.last_id] = 20.0
            return out
    c.ok(generate(Stub(mapping["爱"]), vocab, "我", max_new_tokens=1)["completion"] == "爱", "生成应取末位置 logits")
    eos_result = generate(Stub(EOS_ID), vocab, "我", max_new_tokens=5)
    c.ok(eos_result["stop_reason"] == "eos" and eos_result["generated_ids"] == [EOS_ID], "EOS 必须立即停止")
    limit_result = generate(Stub(mapping["爱"]), vocab, "我", max_new_tokens=10)
    c.ok(limit_result["stop_reason"] == "context_limit" and len(limit_result["generated_ids"]) == 3, "上下文限制处理错误")
    sample1 = generate(model, vocab, "我", max_new_tokens=4, sample=True, seed=42)
    sample2 = generate(model, vocab, "我", max_new_tokens=4, sample=True, seed=42)
    c.ok(sample1 == sample2, "同设备、同种子采样应复现")

    # 保存/重载与小批过拟合冒烟实验
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory)/"check.pt"
        save_checkpoint(path, model, vocab, 3)
        loaded, vocab2, meta = load_checkpoint(path, device)
        c.ok(vocab2 == vocab and meta["best_epoch"] == 3, "词表与选择轮次未保存")
        model.eval()
        c.close(loaded(inputs), model(inputs), "重载结果变化", atol=0, rtol=0)
        c.ok(generate(loaded, vocab, "我", max_new_tokens=3) == generate(model, vocab, "我", max_new_tokens=3), "重载后生成变化")
        set_seed(173)
        small = TinyGPT(GPTConfig(len(vocab), max_len=16, d_model=16, n_heads=4,
                                  n_layers=1, ff_dim=32, dropout=0.0)).to(device)
        tiny_data = CharDataset(["我爱学习。"], mapping, 16)
        tiny_loader = make_loader(tiny_data, 1)
        initial = evaluate(small, tiny_loader, device)["loss"]
        opt = torch.optim.AdamW(small.parameters(), lr=0.02, weight_decay=0)
        for _ in range(45):
            train_epoch(small, tiny_loader, opt, device)
        final = evaluate(small, tiny_loader, device)["loss"]
        c.ok(final < initial * 0.1, "单样本拟合未成功，训练链路需检查")
    print(f"通过固定检查：{c.count} 项（{device}）")
    return c.count

In [25]:
CHECK_COUNT = run_checks(DEVICE)
print("自动检查数量：", CHECK_COUNT)

通过固定检查：81 项（cpu）
自动检查数量： 81


# 独立练习区

**先停止向下看，自己完成三个函数，再打开各自的运行开关。**
可以使用上面的基础类和函数，但不要直接调用被要求重写的同名高层函数。
参考答案放在最后；它们不会自动替你填入练习函数。

## 练习 A：补全完整 GPT 的 forward

实现 `gpt_forward_ex(model, input_ids)`，不调用 `model(input_ids)` 或 `model.forward(...)`。
输入已保证符合本课模型约定，使用 `attention_allowed()` 构造 mask，按顺序调用已有子模块。
输出原始 `[B,T,V]` logits，不能 softmax，也不能 detach。

验收比较两层模型的输出，以及所有参数的梯度，防止“形状正确却漏算了一层”。

In [26]:
def gpt_forward_ex(model, input_ids):
    raise NotImplementedError("由你补全 GPT 的逐层前向计算")


def check_forward_answer(function):
    set_seed(1717)
    cfg = GPTConfig(9, max_len=8, d_model=8, n_heads=2, n_layers=2, ff_dim=16, dropout=0.0)
    expected_model = TinyGPT(cfg)
    actual_model = TinyGPT(cfg)
    actual_model.load_state_dict(expected_model.state_dict())
    ids = torch.tensor([[BOS_ID, 4, 5, PAD_ID], [BOS_ID, 6, 7, 4]])
    expected = expected_model(ids)
    actual = function(actual_model, ids)
    torch.testing.assert_close(actual, expected)
    expected.square().mean().backward()
    actual.square().mean().backward()
    for (name, p), (_, q) in zip(expected_model.named_parameters(), actual_model.named_parameters()):
        if q.grad is None:
            raise AssertionError(f"参数 {name} 没有梯度")
        torch.testing.assert_close(q.grad, p.grad, msg=name)
    print("练习 A 检查通过：输出与全部参数梯度一致。")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_forward_answer(gpt_forward_ex)
else:
    print("练习 A 尚未启动：补全后把开关改为 True。")

练习 A 尚未启动：补全后把开关改为 True。


## 练习 B：独立实现一个训练步

实现 `train_step_ex(model, batch, optimizer)`，使用 `loss_parts()` 计算有效目标平均损失，
梯度范数阈值设为 1.0。返回**更新前的损失浮点数**。

按顺序：进入训练模式 → 取同设备 batch → 清零 → 前向 → loss → backward → 裁剪 → step。
测试会故意放入旧梯度，以检查你是否真正清零；还会和正确实现比较每个参数的更新结果。

In [27]:
def train_step_ex(model, batch, optimizer):
    raise NotImplementedError("由你补全一次参数更新")


def check_step_answer(function):
    set_seed(1718)
    cfg = GPTConfig(9, max_len=8, d_model=8, n_heads=2, n_layers=1, ff_dim=16, dropout=0.0)
    reference = TinyGPT(cfg)
    candidate = TinyGPT(cfg)
    candidate.load_state_dict(reference.state_dict())
    batch = {"input_ids": torch.tensor([[1, 4, 5], [1, 6, 3]]),
             "targets": torch.tensor([[4, 5, 2], [6, 2, -100]])}
    opt_r = torch.optim.SGD(reference.parameters(), lr=0.1)
    opt_c = torch.optim.SGD(candidate.parameters(), lr=0.1)
    for p in candidate.parameters():
        p.grad = torch.full_like(p, 7.0)
    reference.train()
    opt_r.zero_grad(set_to_none=True)
    total, count = loss_parts(reference(batch["input_ids"]), batch["targets"])
    loss = total / count
    loss.backward()
    nn.utils.clip_grad_norm_(reference.parameters(), 1.0, error_if_nonfinite=True)
    opt_r.step()
    got = function(candidate, batch, opt_c)
    if not isinstance(got, float) or abs(got-loss.item()) > 1e-6:
        raise AssertionError("必须返回更新前的平均损失浮点数")
    for (name, p), (_, q) in zip(reference.named_parameters(), candidate.named_parameters()):
        torch.testing.assert_close(q, p, msg=name)
    print("练习 B 检查通过：损失、清零、裁剪和参数更新一致。")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_step_answer(train_step_ex)
else:
    print("练习 B 尚未启动：补全后把开关改为 True。")

练习 B 尚未启动：补全后把开关改为 True。


## 练习 C：独立实现贪心生成

实现 `greedy_ids_ex(model, prefix_ids, max_new_tokens=5)`。本题输入是已包含 BOS、无 PAD、同设备的 `[1,T]` 编号，
已保证合法且长度不超过上下文。返回包含原前缀与新增编号的张量，不改变传入张量或模型参数。

要求：eval + no_grad、只用末位置 logits、禁止 BOS/PAD、逐次追加，遇 EOS 停止，
不能把超过上下文的输入送进模型，结束时恢复原整体模式。本题不做采样和字符串解码。

固定检查使用可预测的假模型，专门验证循环；不能用“生成得通顺”替代这些检查。

In [28]:
def greedy_ids_ex(model, prefix_ids, max_new_tokens=5):
    raise NotImplementedError("由你补全自回归贪心循环")


def check_generation_answer(function):
    class PredictableModel(nn.Module):
        def __init__(self, last_id):
            super().__init__()
            self.anchor = nn.Parameter(torch.zeros(1))
            self.cfg = GPTConfig(9, max_len=4, d_model=8, n_heads=2, n_layers=1, ff_dim=16)
            self.last_id = last_id
        def forward(self, ids):
            if ids.shape[1] > self.cfg.max_len:
                raise AssertionError("超长输入被送进了模型")
            out = torch.full((*ids.shape, 9), -20.0)
            out[:, :, 5] = 10.0
            out[:, -1, :] = -20.0
            out[:, -1, self.last_id] = 10.0
            # 两个禁用符号得分更高：生成函数必须主动屏蔽。
            out[:, -1, [BOS_ID, PAD_ID]] = 100.0
            return out
    prefix = torch.tensor([[BOS_ID, 4]])
    saved_prefix = prefix.clone()
    m = PredictableModel(6)
    m.train()
    got = function(m, prefix, max_new_tokens=1)
    torch.testing.assert_close(got, torch.tensor([[BOS_ID, 4, 6]]))
    if not m.training:
        raise AssertionError("没有恢复调用前模式")
    torch.testing.assert_close(prefix, saved_prefix)
    torch.testing.assert_close(function(m, prefix, 0), prefix)
    eos = function(PredictableModel(EOS_ID), prefix, 9)
    torch.testing.assert_close(eos, torch.tensor([[BOS_ID, 4, EOS_ID]]))
    limited = function(m, prefix, 9)
    if limited.shape != (1, 5):
        raise AssertionError("上下文终止边界不正确")
    if m.anchor.grad is not None:
        raise AssertionError("生成不应产生参数梯度")
    print("练习 C 检查通过：末位置、禁用符、EOS、上下文、零步和模式恢复。")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_generation_answer(greedy_ids_ex)
else:
    print("练习 C 尚未启动：补全后把开关改为 True。")

练习 C 尚未启动：补全后把开关改为 True。


---
# 参考答案

先完成上面的练习再核对。以下答案使用了前文定义的类、基础函数和练习检查器，
但不调用你尚未完成的练习函数。单独的 `.py` 脚本则包含可直接运行的完整实现。

## A. 完整 forward 参考实现

In [29]:
def gpt_forward_reference(model, input_ids):
    length = input_ids.shape[1]
    positions = torch.arange(length, device=input_ids.device)
    allowed = attention_allowed(input_ids != PAD_ID)
    x = model.token_embedding(input_ids) + model.position_embedding(positions)[None]
    x = model.embedding_dropout(x)
    for block in model.blocks:
        x = block(x, allowed)
    return model.lm_head(model.final_norm(x))

check_forward_answer(gpt_forward_reference)

练习 A 检查通过：输出与全部参数梯度一致。


## B. 单步训练参考实现

In [30]:
def train_step_reference(model, batch, optimizer):
    model.train()
    device = next(model.parameters()).device
    inputs = batch["input_ids"].to(device)
    targets = batch["targets"].to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    total, count = loss_parts(logits, targets)
    loss = total / count
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    optimizer.step()
    return loss.detach().item()

check_step_answer(train_step_reference)

练习 B 检查通过：损失、清零、裁剪和参数更新一致。


## C. 贪心生成参考实现

In [31]:
def greedy_ids_reference(model, prefix_ids, max_new_tokens=5):
    ids = prefix_ids.clone()
    was_training = model.training
    try:
        model.eval()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                if ids.shape[1] > model.cfg.max_len:
                    break
                last_logits = model(ids)[:, -1, :].clone()
                last_logits[:, [BOS_ID, PAD_ID]] = float("-inf")
                next_id = last_logits.argmax(dim=-1, keepdim=True)
                ids = torch.cat([ids, next_id], dim=1)
                if int(next_id.item()) == EOS_ID:
                    break
    finally:
        model.train(was_training)
    return ids

check_generation_answer(greedy_ids_reference)

练习 C 检查通过：末位置、禁用符、EOS、上下文、零步和模式恢复。


## 命令行使用：不需要打开 Notebook

把脚本放进项目 `lessons` 目录，或直接在脚本所在目录执行：

```bash
python lesson_17_tiny_gpt_pretraining.py
python lesson_17_tiny_gpt_pretraining.py --checks-only
python lesson_17_tiny_gpt_pretraining.py --device auto
```

自己的 UTF-8 文件，每行一句，至少 5 条不同的非空句子：

```bash
python lesson_17_tiny_gpt_pretraining.py --text-file my_text.txt --output-dir my_run_17
```

每句字符数加 BOS 后不能超过 `--max-len`。超长句请先手动分段，本课不静默截断。
字符词表遇到大量生僻字符会扩大，因此先从几十到几百条短文本验证流程，别直接导入巨量数据。

输入文件缺失会报错；只有默认演示文件会自动创建。所有相对路径以**当前工作目录**为准。
重复运行输出会覆盖；不修改 `.py` 源码或 Notebook 练习答案。

| 输出文件 | 含义 |
|---|---|
| `split.json` | 实际训练/验证文本与训练词表 |
| `metrics.json` | 环境、配置、数据摘要、每轮指标、最佳轮次、重载检查 |
| `samples.json` | 提示、生成编号、文本与终止原因 |
| `best_model.pt` | 配置、词表、选定的模型权重；不是完整续训状态 |

## 常见问题与排查

| 现象 | 首先检查 |
|---|---|
| 刚开始 loss 就异常小 | 标签是否等于当前输入？是否漏掉因果 mask？ |
| 整个序列输出都是 NaN | 是否有一整行 attention 被全部屏蔽？梯度/学习率是否异常？ |
| 所有句子只生成一种模板 | 语料是否单一？贪心是否总选同一个并列高概率分支？ |
| 保存后重载乱码 | 词表顺序是否完全相同？模型配置是否匹配？ |
| 有效长度变了，验证 loss 就变 | 是否按有效 token 加权？是否误把 PAD 算进目标？ |
| 调高 max_len 就认为模型学会长文本 | 位置表容量只定义可处理的形状；没见过的位置并不自动学好 |
| GPU 与 CPU 数字不同 | 随机数/数值内核/库版本可能不同，别要求跨设备逐位一致 |
| 验证比训练差 | 先查数据与统计；差距本身不等于代码错误，也不必无限训练 |

安全边界：只运行自己理解或信任的代码，只加载信任的检查点。不要把含凭据或私人信息的文件当作随意分享的语料。

## 原始资料与官方接口

[1] [PyTorch ModuleList](https://docs.pytorch.org/docs/stable/generated/torch.nn.ModuleList.html)：注册和遍历子模块。  
[2] [PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)：logits、整数标签和忽略目标。  
[3] [PyTorch SDPA](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)：因果/布尔 mask 与 Dropout 约定。  
[4] [PyTorch AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html)：优化器和解耦权重衰减。  
[5] [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)：梯度范数裁剪。  
[6] [保存与加载模型](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)：参数字典、配置和恢复方式。  
[7] [原始 GPT 研究代码](https://github.com/openai/finetune-transformer-lm)：历史原始实现，用于理解技术来源；本课无需安装。  
[8] [PyTorch Autograd 说明](https://docs.pytorch.org/docs/stable/notes/autograd.html)：评估模式与关闭梯度的区别。  
[9] [PyTorch 可复现性说明](https://docs.pytorch.org/docs/stable/notes/randomness.html)：种子与跨环境复现边界。  
[10] [Hugging Face Perplexity](https://huggingface.co/docs/transformers/perplexity)：PPL 定义与分词、上下文对指标的影响。

上述资料解释的是接口或方法；本讲义的参数数量、训练数值、检查结论来自本地实际运行，不能推广成通用性能承诺。

## 本课验收与下一步

关掉参考答案后，独立完成 A、B、C，并能口述：

> 数据标签移位一次；完整模型产生每个位置的词表分数；因果约束阻止读取未来；
> 交叉熵只统计有效目标；反向传播更新参数；生成只在固定权重下不断追加 token。

再做一个变化实验：保持数据和随机种子不变，用 1 层、2 层模型分别训练，保存到不同目录。
记录参数量、训练/验证损失和样例，不要因为样例更流畅就认定所有能力更强。

**下一课：第 18 课——解码策略：温度、Top-k、Top-p 与生成评测。**
我们会固定同一模型，研究为什么不同解码规则产生不同输出，再区分模型能力与解码策略的作用。